# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# The source file is Windows-1252 encoded, not UTF-8, and the default read
# fails on a curly apostrophe, byte 0x92
df = pd.read_csv('data/AviationData.csv', encoding='cp1252', low_memory=False)
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df.info()

# USState_Codes.csv is loaded further down, to derive a State column from Location

(df.isna().mean() * 100).round(1).sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                88889 non-null  object        
 1   Investigation.Type      88889 non-null  object        
 2   Accident.Number         88889 non-null  object        
 3   Event.Date              88889 non-null  datetime64[ns]
 4   Location                88837 non-null  object        
 5   Country                 88663 non-null  object        
 6   Latitude                34382 non-null  object        
 7   Longitude               34373 non-null  object        
 8   Airport.Code            50132 non-null  object        
 9   Airport.Name            52704 non-null  object        
 10  Injury.Severity         87889 non-null  object        
 11  Aircraft.damage         85695 non-null  object        
 12  Aircraft.Category       32287 non-null  object

Schedule                  85.8
Air.carrier               81.3
FAR.Description           64.0
Aircraft.Category         63.7
Latitude                  61.3
Longitude                 61.3
Airport.Code              43.6
Airport.Name              40.7
Broad.phase.of.flight     30.6
Publication.Date          15.5
Total.Serious.Injuries    14.1
Total.Minor.Injuries      13.4
Total.Fatal.Injuries      12.8
Engine.Type                8.0
Report.Status              7.2
Purpose.of.flight          7.0
Number.of.Engines          6.8
Total.Uninjured            6.7
Weather.Condition          5.1
Aircraft.damage            3.6
Registration.Number        1.6
Injury.Severity            1.1
Country                    0.3
Model                      0.1
Amateur.Built              0.1
Make                       0.1
Location                   0.1
Investigation.Type         0.0
Event.Date                 0.0
Accident.Number            0.0
Event.Id                   0.0
dtype: float64

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# --- Impute Aircraft.Category from Make + Model where unambiguous ---
make_model_key = (df['Make'].str.strip().str.upper() + ' '
                  + df['Model'].str.strip().str.upper())

labelled = df['Aircraft.Category'].notna()
category_lut = (
    df[labelled]
      .groupby(make_model_key[labelled])['Aircraft.Category']
      .agg(['nunique', 'first'])
      .query('nunique == 1')['first']      # only airframes with one consistent category
)

missing_before = df['Aircraft.Category'].isna().sum()
df['Aircraft.Category'] = df['Aircraft.Category'].fillna(make_model_key.map(category_lut))
recovered = missing_before - df['Aircraft.Category'].isna().sum()
print(f'Category blanks: {missing_before:,} | recovered by Make+Model lookup: {recovered:,} '
      f'| still unknown: {df["Aircraft.Category"].isna().sum():,}\n')

# --- Filter to the aircraft and events the client cares about ---
steps = {
    'raw': pd.Series(True, index=df.index),
    'event year >= 1983': df['Event.Date'].dt.year >= 1983,
    'not amateur built': df['Amateur.Built'].str.strip().str.lower() == 'no',
    'accidents (not incidents)': df['Investigation.Type'] == 'Accident',
    'airplanes only': df['Aircraft.Category'] == 'Airplane',
}

mask = pd.Series(True, index=df.index)
for label, condition in steps.items():
    mask &= condition
    print(f'{label:<28} {mask.sum():>6,}')

df = df[mask].copy()

print(
    f'\nrows: {len(df):,} | '
    f'years: {df["Event.Date"].dt.year.min()}-'
    f'{df["Event.Date"].dt.year.max()} | '
    f'categories: {df["Aircraft.Category"].unique()}'
)

# Confirm the sample is spread across the whole window, not bunched in recent years
decade_bins = [1982, 1990, 2000, 2008, 2015, 2023]
print('\nrows per era:')
print(df.groupby(pd.cut(df['Event.Date'].dt.year, decade_bins),
                 observed=True).size().to_string())

Category blanks: 56,602 | recovered by Make+Model lookup: 44,360 | still unknown: 12,242

raw                          88,889
event year >= 1983           85,289
not amateur built            76,960
accidents (not incidents)    73,286
airplanes only               58,659

rows: 58,659 | years: 1983-2022 | categories: ['Airplane']

rows per era:
Event.Date
(1982, 1990]    16879
(1990, 2000]    15541
(2000, 2008]    10469
(2008, 2015]     8044
(2015, 2023]     7726


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
# --- Injury columns: clean, impute, and derive the key injury measure ---

inj_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
            'Total.Minor.Injuries', 'Total.Uninjured']

# Injury.Severity encodes the same as text: 'Fatal(N)' gives the death
# count, and 'Non-Fatal'/'Incident'/'Minor'/'Serious' imply zero deaths.
sev = df['Injury.Severity'].astype(str).str.strip()
fatal_from_sev = sev.str.extract(r'Fatal\((\d+)\)')[0].astype(float)
fatal_from_sev[sev.isin(['Non-Fatal', 'Incident', 'Minor', 'Serious'])] = 0
df['Total.Fatal.Injuries'] = df['Total.Fatal.Injuries'].fillna(fatal_from_sev)

# Assumption: a remaining blank in these tallies means "none in this category"
# Fill with 0 so the counts add up
df[inj_cols] = df[inj_cols].fillna(0)

df['Total.Occupants'] = df[inj_cols].sum(axis=1)

# Fatal or serious injury, as a rate per occupant.
# Deliberately left NaN where every injury tally was blank, so that occupant
# count is 0 and the rate is undefined. This keeps them out of mean injury rates 
# instead of counting them as accidents where nobody was hurt.
df['Fatal.Serious.Injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
df['Fatal.Serious.Rate'] = (
    df['Fatal.Serious.Injuries'] / df['Total.Occupants']
).where(df['Total.Occupants'] > 0)

print(f"Accidents with no recorded occupants (rate left NaN): "
      f"{(df['Total.Occupants'] == 0).sum():,}")

df[['Total.Occupants', 'Fatal.Serious.Injuries', 'Fatal.Serious.Rate']].describe()

Accidents with no recorded occupants (rate left NaN): 321


,Total.Occupants,Fatal.Serious.Injuries,Fatal.Serious.Rate
count,58659.000000,58659.000000,58338.000000
mean,4.363934,0.806117,0.277936
std,20.889290,5.249024,0.432172
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,0.800000
max,576.000000,295.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
print(df['Aircraft.damage'].value_counts(dropna=False))

# Cleaning tasks: strip whitespace, capitalize, replace 'Unknown' with np.nan
df['Aircraft.damage'] = (
    df['Aircraft.damage']
      .str.strip()
      .str.title()
      .replace({'Unknown': np.nan})
)

# Do not drop rows with unrecorded damage. Instead, leave Destroyed as NaN for these rows. 
# Coercing to 0 would assert "not destroyed" when we don't know what happened. 
# As NaN, these rows are skipped by .mean() when computing destruction rates, 
# but we can still use their injury data.
unknown_damage = df['Aircraft.damage'].isna()
print(f"\nRows with unrecorded damage (kept, Destroyed = NaN): {unknown_damage.sum():,}")
print(f"  their mean occupancy: {df.loc[unknown_damage, 'Total.Occupants'].mean():.1f} "
      f"vs {df.loc[~unknown_damage, 'Total.Occupants'].mean():.1f} for the rest")

df['Destroyed'] = np.where(
    unknown_damage, np.nan, df['Aircraft.damage'] == 'Destroyed'
).astype(float)

# Create ordinal encoding
damage_order = ['Minor', 'Substantial', 'Destroyed']
df['Damage.Severity'] = pd.Categorical(
    df['Aircraft.damage'], categories=damage_order, ordered=True
)

print()
print(df['Aircraft.damage'].value_counts(dropna=False))
print(f"\nOverall destruction rate (known-damage rows only): {df['Destroyed'].mean():.1%}")

Aircraft.damage
Substantial    45309
Destroyed      12010
NaN              844
Minor            421
Unknown           75
Name: count, dtype: int64

Rows with unrecorded damage (kept, Destroyed = NaN): 919
  their mean occupancy: 66.3 vs 3.4 for the rest

Aircraft.damage
Substantial    45309
Destroyed      12010
NaN              919
Minor            421
Name: count, dtype: int64

Overall destruction rate (known-damage rows only): 20.8%


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
print(f"Rows: {len(df):,}")
print(f"Missing Make: {df['Make'].isna().sum()} ({df['Make'].isna().mean():.2%})")

# Task 1: strip whitespace, uppercase, collapse runs of internal spaces
df['Make'] = df['Make'].str.strip().str.upper().str.replace(r'\s+', ' ', regex=True)
print(f"Unique after strip + upper: {df['Make'].nunique():,}")

# How badly is a single manufacturer fragmented before consolidation?
for brand in ['CESSNA', 'BOEING', 'PIPER', 'BEECH', 'BOMBARDIER']:
    variants = sorted(v for v in df['Make'].dropna().unique() if brand in v)
    print(f"\n{brand}: {len(variants)} distinct spellings")
    print(variants[:12])

# --- Tasks 2 and 3: ordered, first-match-wins brand patterns ---------------
# Order matters: the more specific brand must be tested before the more general
# one, or it gets swallowed. AMERICAN CHAMPION before CHAMPION, MCDONNELL
# before DOUGLAS, ROCKWELL before NORTH AMERICAN, and GRUMMAN before the bare
# AMERICAN AA-1/AA-5 airframes.
BRANDS = [
    (r'CUB ?CRAFTERS',                      'CUB CRAFTERS'),
    (r'CESSNA',                             'CESSNA'),
    (r'PIPER',                              'PIPER'),
    # Beech's successors: Raytheon owned Beech 1994-2006, then Hawker Beechcraft.
    # Their models here are Beech airframes (A36, B200, C90A, 58).
    (r'HAWKER[ -]BEECH|RAYTHEON|BEECH',     'BEECH'),
    (r'BOEING',                             'BOEING'),
    (r'AIRBUS',                             'AIRBUS'),
    (r'MOONEY',                             'MOONEY'),
    (r'MCDONNELL',                          'MCDONNELL DOUGLAS'),
    (r'DOUGLAS',                            'DOUGLAS'),
    (r'AMERICAN CHAMPION',                  'AMERICAN CHAMPION'),
    (r'CHAMPION',                           'CHAMPION'),
    (r'ROCKWELL',                           'ROCKWELL'),
    (r'NORTH AMERICAN',                     'NORTH AMERICAN'),
    # All the G-164 Ag-Cat variants are one airframe: Grumman designed it,
    # Schweizer built it, Gulfstream American owned the line for a while.
    # The bare AMERICAN rows are the AA-1/AA-5, i.e. Grumman American.
    (r'GRUMMAN|SCHWEIZER|^AMERICAN AVIATION|^AMERICAN$|AMERICAN GENERAL',
                                            'GRUMMAN'),
    (r'GULFSTREAM',                         'GULFSTREAM'),
    (r'CIRRUS',                             'CIRRUS'),
    (r'AIR TRACTOR',                        'AIR TRACTOR'),
    # Anchored: a bare 'AVIAT' substring also matches every '<BRAND> AVIATION',
    # which pulled Dassault, Textron, Reims, Scottish, Weatherly and Taylorcraft
    # airframes into this make.
    (r'^AVIAT\b|AVIAT AIRCRAFT',            'AVIAT'),
    (r'AYRES',                              'AYRES'),
    (r'DIAMOND AIC?RAFT|^DIAMOND',          'DIAMOND'),   # 'AICRAFT' typo in source
    (r'EMBRAER',                            'EMBRAER'),
    (r'SOCATA',                             'SOCATA'),
    (r'ERCOUPE',                            'ERCOUPE'),
    (r'BOMBARDIER|CANADAIR',                'BOMBARDIER'),  # Canadair became Bombardier in 1986
    (r'LEARJET|LEAR JET',                   'LEARJET'),     # Gates Learjet, same airframes
    (r'DE ?HAVILLAND',                      'DE HAVILLAND'),
    (r'MAULE',                              'MAULE'),
    (r'LUSCOMBE',                           'LUSCOMBE'),
    (r'STINSON',                            'STINSON'),
    (r'TAYLORCRAFT',                        'TAYLORCRAFT'),
    (r'AERONCA',                            'AERONCA'),
    (r'GREAT LAKES',                        'GREAT LAKES'),
    (r'WACO',                               'WACO'),
    (r'^LAKE',                              'LAKE'),
    (r'^HELIO',                             'HELIO'),
    (r'PZL',                                'PZL'),
    (r'^RYAN',                              'RYAN'),
    (r'AERO COMMANDER',                     'AERO COMMANDER'),
    (r'WEATHERLY',                          'WEATHERLY'),
    (r'FLIGHT DESIGN',                      'FLIGHT DESIGN'),
    (r'EXTRA FLUGZEUG|^EXTRA',              'EXTRA'),
    (r'PITTS',                              'PITTS'),
    (r'SWEARINGEN',                         'SWEARINGEN'),
    (r'FAIRCHILD',                          'FAIRCHILD'),
    (r'LOCKHEED',                           'LOCKHEED'),
    (r'MITSUBISHI',                         'MITSUBISHI'),
    (r'^GLOBE',                             'GLOBE'),
    (r'NAVION',                             'NAVION'),
    (r'CHRISTEN',                           'CHRISTEN'),
    (r'SAAB',                               'SAAB'),
]

# Fallback for the long tail: drop parentheticals and corporate suffixes so
# 'X AIRCRAFT CO.' and 'X INC' collapse onto 'X' without a hand-written entry.
SUFFIX = (r'\s*\b(AIRCRAFT|AIRPLANE|AIRPLANES|AVIATION|AVN|ACFT|AEROSPACE|'
          r'CORPORATION|CORP|COMPANY|CO|INCORPORATED|INC|LTD|LIMITED|LLC|LP|'
          r'GMBH|AG|SA|AB|AS|BV|NV|SRO|SPOL|PTY|SPA|'
          r'INTERNATIONAL|INDUSTRIES|INDUSTRY|IND|GROUP|HOLDINGS|MFG|DIV|DIVISION'
          r')\b\.?')

make = df['Make']
canonical = pd.Series(np.nan, index=df.index, dtype=object)
for pattern, name in BRANDS:
    hit = canonical.isna() & make.str.contains(pattern, regex=True, na=False)
    canonical[hit] = name

tail = canonical.isna() & make.notna()
canonical[tail] = (
    make[tail]
      .str.replace(r'\([^)]*\)', ' ', regex=True)
      .str.replace(SUFFIX, ' ', regex=True)
      .str.replace(r'[.,]+', ' ', regex=True)
      .str.replace(r'\s+', ' ', regex=True)
      .str.strip(' -/')
      .replace('', np.nan)
)
df['Make'] = canonical

# Task 4: rows with no Make cannot be attributed to a manufacturer
df = df.dropna(subset=['Make']).copy()
print(f"\nUnique after consolidation: {df['Make'].nunique():,}")

# The brands that were worst fragmented, now consolidated
for brand in ['BOMBARDIER', 'CESSNA', 'CIRRUS', 'GRUMMAN', 'BEECH']:
    print(f"  {brand}: {(df['Make'] == brand).sum():,} accidents")

# Task 5: keep makes with a usable sample so per-make rates mean something
MIN_ACCIDENTS_PER_MAKE = 50
make_counts = df['Make'].value_counts()
keep = make_counts[make_counts >= MIN_ACCIDENTS_PER_MAKE].index

rows_before = len(df)
df = df[df['Make'].isin(keep)].copy()
print(f"\nMakes with >= {MIN_ACCIDENTS_PER_MAKE} accidents: {len(keep)} of {len(make_counts)}")
print(f"Rows: {rows_before:,} -> {len(df):,} ({len(df) / rows_before:.1%} retained)")

display(df['Make'].value_counts().to_frame('accidents'))

Rows: 58,659
Missing Make: 2 (0.00%)
Unique after strip + upper: 1,064

CESSNA: 11 distinct spellings
['CESSNA', 'CESSNA AIRCRAFT', 'CESSNA AIRCRAFT CO', 'CESSNA AIRCRAFT CO.', 'CESSNA AIRCRAFT COMPANY', 'CESSNA ECTOR', 'CESSNA REIMS', 'CESSNA SKYHAWK II', 'CESSNA/AIR REPAIR INC', 'CESSNA/WEAVER', 'REIMS-CESSNA']

BOEING: 7 distinct spellings
['BOEING', 'BOEING (STEARMAN)', 'BOEING COMPANY', 'BOEING OF CANADA/DEHAV DIV', 'BOEING STEARMAN', 'BOEING-STEARMAN', 'THE BOEING COMPANY']

PIPER: 15 distinct spellings
['JETPROP DLX PIPER', 'NEW PIPER', 'NEW PIPER AIRCRAFT INC', 'PIPER', 'PIPER / LAUDEMAN', 'PIPER AEROSTAR', 'PIPER AIRCRAFT', 'PIPER AIRCRAFT CORPORATION', 'PIPER AIRCRAFT INC', 'PIPER AIRCRAFT, INC.', 'PIPER CUB CRAFTERS', 'PIPER PAWNEE']

BEECH: 14 distinct spellings
['BEECH', 'BEECH AIRCRAFT', 'BEECH AIRCRAFT CO.', 'BEECH AIRCRAFT CORP', 'BEECH AIRCRAFT CORPORATION', 'BEECHCRAFT', 'BEECHCRAFT CORPORATION', 'HAWKER BEECH', 'HAWKER BEECHCRAFT', 'HAWKER BEECHCRAFT CORP', 'HAWKER B

,accidents
Make,
CESSNA,24890
PIPER,13597
BEECH,4567
GRUMMAN,1674
MOONEY,1227
BOEING,1102
BELLANCA,915
AIR TRACTOR,855
AERONCA,536


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
print(f"Missing Model values: {df['Model'].isna().sum()}")

# Drop NaNs
df = df.dropna(subset=['Model']).copy()

# Normalize case/whitespace
df['Model'] = df['Model'].str.strip().str.upper()

print(f"Unique models after normalizing: {df['Model'].nunique()}")
display(df['Model'].value_counts().head(15))

# Are model labels unique to each make?
makes_per_model = df.groupby('Model')['Make'].nunique().sort_values(ascending=False)
shared = makes_per_model[makes_per_model > 1]
print(f"{len(shared)} of {len(makes_per_model)} model labels appear under more than one Make")
display(shared.head(10))

# Build a combined identifier Make_Model
df['Make_Model'] = df['Make'] + ' ' + df['Model']
print(f"Unique Make_Model plane types: {df['Make_Model'].nunique()}")
display(df['Make_Model'].value_counts().head(15))

Missing Model values: 12
Unique models after normalizing: 2337


Model
152          2216
172          1641
172N         1092
PA-28-140     862
172M          754
150           745
172P          661
182           611
180           594
PA-18-150     556
PA-18         552
150M          549
PA-28-180     546
PA-28-161     523
PA-28-181     508
Name: count, dtype: int64

79 of 2337 model labels appear under more than one Make


Model
DC-10-30    3
7GCBC       3
NAVION A    3
S2R         3
8KCAB       3
8GCBC       3
500         3
7KCAB       3
7GCB        3
7GCAA       3
Name: Make, dtype: int64

Unique Make_Model plane types: 2431


Make_Model
CESSNA 152         2216
CESSNA 172         1641
CESSNA 172N        1092
PIPER PA-28-140     862
CESSNA 172M         754
CESSNA 150          745
CESSNA 172P         661
CESSNA 182          611
CESSNA 180          594
PIPER PA-18         550
PIPER PA-18-150     549
CESSNA 150M         549
PIPER PA-28-180     546
PIPER PA-28-161     523
PIPER PA-28-181     508
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
other_cols = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
              'Purpose.of.flight', 'Broad.phase.of.flight']

print('BEFORE')
for col in other_cols:
    print(f'\n{col}')
    print(df[col].value_counts(dropna=False))

placeholders = ['Unknown', 'Unk', 'UNK', 'Other', 'None', '']

# strip / title-case Engine.Type, Purpose.of.flight, and Broad.phase.of.flight
# replace placeholders with NaN
for col in ['Engine.Type', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    df[col] = df[col].str.strip().str.title().replace(placeholders, np.nan)

# Uppercase the Weather.Condition acronym
df['Weather.Condition'] = (
    df['Weather.Condition'].str.strip().str.upper().replace(placeholders, np.nan)
)

# For Purpose.of.flight, collapse labels that mean the same thing
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Air Race/Show': 'Air Race Show',
    # the four Public Aircraft levels differ only by which government owns the plane
    'Public Aircraft - Federal': 'Public Aircraft',
    'Public Aircraft - State': 'Public Aircraft',
    'Public Aircraft - Local': 'Public Aircraft',
    # undocumented NTSB codes, setting to NaN
    'Asho': np.nan,
    'Pubs': np.nan,
})

BEFORE

Engine.Type
Engine.Type
Reciprocating    49358
NaN               3233
Turbo Prop        2148
Turbo Fan          951
Unknown            400
Turbo Jet          199
Turbo Shaft         12
UNK                  1
Name: count, dtype: int64

Weather.Condition
Weather.Condition
VMC    49279
IMC     4401
NaN     1973
UNK      491
Unk      158
Name: count, dtype: int64

Number.of.Engines
Number.of.Engines
1.0    46751
2.0     6898
NaN     2380
3.0      100
0.0       90
4.0       83
Name: count, dtype: int64

Purpose.of.flight
Purpose.of.flight
Personal                     33456
Instructional                 7862
Aerial Application            3222
Unknown                       2903
NaN                           2839
Business                      2668
Positioning                    903
Other Work Use                 496
Ferry                          478
Aerial Observation             373
Executive/corporate            289
Public Aircraft                252
Skydiving                      1

In [9]:
# --- Derive State from Location ---

state_codes = pd.read_csv('data/USState_Codes.csv')

# The lookup file is not purely states: the last three entries are the NTSB's
# over-water codes (Gulf of Mexico, Atlantic ocean, Pacific ocean). Those are not
# things we can compare against a state, and they cover 30 rows,
# so exclude them and let those accidents fall through to NaN.
OVER_WATER = {'GM', 'AO', 'PO'}
valid_abbrev = set(state_codes['Abbreviation']) - OVER_WATER

# Location is formatted 'CITY, ST'. Take the two-letter code, then keep
# only codes that are genuine US state abbreviations.
abbrev = (
    df['Location'].astype(str).str.strip()
      .str.extract(r',\s*([A-Za-z]{2})\s*$')[0]
      .str.upper()
)
df['State'] = abbrev.where(abbrev.isin(valid_abbrev))
df['State.Name'] = df['State'].map(state_codes.set_index('Abbreviation')['US_State'])

# Non-US, over-water and unparseable rows are left NaN instead of dropped
is_us = df['Country'].str.strip() == 'United States'
print(f"Rows: {len(df):,} | US rows: {is_us.sum():,} ({is_us.mean():.1%})")
print(f"State parsed for {df['State'].notna().mean():.1%} of all rows, "
      f"{df.loc[is_us, 'State'].notna().mean():.1%} of US rows")
print(f"Non-US rows that picked up a state (should be zero): "
      f"{df.loc[~is_us, 'State'].notna().sum()}")

print('\nTop 10 states by accident count:')
print(df['State'].value_counts().head(10).to_string())

# Outcome rates vary widely by operating environment,
# so a make concentrated in one state inherits that state's rate.
by_state = df.groupby('State')['Destroyed'].agg(['mean', 'size'])
by_state = by_state[by_state['size'] >= 300].sort_values('mean')
print(f"\nDestruction rate by state ({len(by_state)} states with >= 300 accidents), "
      f"five lowest and five highest:")
display(
    pd.concat([by_state.head(5), by_state.tail(5)])
      .rename(columns={'mean': 'destroyed rate', 'size': 'accidents'})
      .style.format({'destroyed rate': '{:.1%}'})
)

Rows: 56,302 | US rows: 53,182 (94.5%)
State parsed for 94.3% of all rows, 99.9% of US rows
Non-US rows that picked up a state (should be zero): 0

Top 10 states by accident count:
State
CA    5437
AK    4537
TX    3708
FL    3578
CO    1731
AZ    1720
WA    1572
GA    1367
MI    1353
IL    1229

Destruction rate by state (44 states with >= 300 accidents), five lowest and five highest:


,destroyed rate,accidents
State,,
AK,8.8%,4537
ND,14.1%,405
IA,14.4%,520
MS,17.1%,577
MO,17.1%,1034
MA,23.3%,651
LA,23.5%,748
VA,24.1%,814
CA,25.1%,5437


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [10]:
# Drop columns that are mostly empty or carry no signal

print(f"Shape before: {df.shape}")
na_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
display(na_pct.to_frame('% NaN'))

NA_THRESHOLD = 50
too_sparse = na_pct[na_pct > NA_THRESHOLD].index.tolist()

# One value each so no signal left
constant = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]

# Clean row identifiers, report metadata, and geographical data that doesn't
# pertain to the make/model question
admin = ['Event.Id', 'Accident.Number', 'Registration.Number',
         'Report.Status', 'Publication.Date',
         'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name']

to_drop = [c for c in dict.fromkeys(too_sparse + constant + admin) if c in df.columns]
print(f"\nToo sparse (>{NA_THRESHOLD}% NaN): {too_sparse}")
print(f"Constant after filtering:  {constant}")
print(f"Identifiers and metadata:    {[c for c in admin if c in df.columns]}")

df = df.drop(columns=to_drop)

print(f"\nDropped {len(to_drop)} columns. Shape after: {df.shape}")
print("\nRemaining NaN %:")
display((df.isna().mean() * 100).round(1).sort_values(ascending=False).to_frame('% NaN'))


Shape before: (56302, 39)


,% NaN
Schedule,89.2
Air.carrier,82.8
FAR.Description,68.1
Latitude,60.7
Longitude,60.7
Airport.Code,39.3
Airport.Name,36.6
Broad.phase.of.flight,29.9
Publication.Date,16.7
Purpose.of.flight,10.2



Too sparse (>50% NaN): ['Schedule', 'Air.carrier', 'FAR.Description', 'Latitude', 'Longitude']
Constant after filtering:  ['Investigation.Type', 'Aircraft.Category', 'Amateur.Built']
Identifiers and metadata:    ['Event.Id', 'Accident.Number', 'Registration.Number', 'Report.Status', 'Publication.Date', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name']

Dropped 15 columns. Shape after: (56302, 24)

Remaining NaN %:


,% NaN
Broad.phase.of.flight,29.9
Purpose.of.flight,10.2
Engine.Type,6.5
State.Name,5.7
State,5.7
Weather.Condition,4.7
Number.of.Engines,4.2
Aircraft.damage,1.6
Damage.Severity,1.6
Destroyed,1.6


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [11]:
# index=False: the row labels here are leftover positions from the raw file and
# carry no meaning, and writing them adds a stray 'Unnamed: 0' column on reload
df.to_csv("CleanedAviationData.csv", index=False)

print(f"Saved {df.shape[0]:,} rows x {df.shape[1]} columns")
print(list(df.columns))

Saved 56,302 rows x 24 columns
['Event.Date', 'Location', 'Country', 'Injury.Severity', 'Aircraft.damage', 'Make', 'Model', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Total.Occupants', 'Fatal.Serious.Injuries', 'Fatal.Serious.Rate', 'Destroyed', 'Damage.Severity', 'Make_Model', 'State', 'State.Name']
